<h1 style="text-align:center; color:#1a1a2; font-size:2.5rem;">🔗 LangChain Data Connections</h1>
<p style="text-align:center; color:#55; font-size:1rem;">A step-by-step pipeline that loads a document, splits it into chunks, converts them to vectors, stores them in a database, and retrieves relevant content based on a question.</p>

<h2 style="color:#35F527;">🗺️ Pipeline Overview</h2>

| Step | Component | What it does |
|---|---|---|
| 1️⃣ | Document Loader | Reads the text file into memory |
| 2️⃣ | Text Splitter | Cuts text into small chunks |
| 3️⃣ | Embeddings | Converts chunks into vectors (numbers) |
| 4️⃣ | Vector Store | Stores vectors in a searchable database |
| 5️⃣ | Retriever | Finds the most relevant chunks for a question |

## 🖼️ Project Overview
![Data Connections Project](Data%20Connections%20-%20Project.PNG)

## Data Connection Flow
![Data Connection Diagram](DataConnection.png)


<h2 style="color:#F527F2;"> Imports</h2>

In [5]:
!pip install langchain-community langchain-text-splitters sentence-transformers chromadb

  Using cached tokenizers-0.22.2-cp39-abi3-macosx_10_12_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_10_12_x86_64.whl.metadata (4.1 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10

In [8]:
# Document loading and splitting
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter

# Embeddings — converts text to vectors
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

<hr>
<h2 style="color:#27EEF5;"> Document Loader</h2>
<p style="color:#27EEF5; font-size:0.95rem;">
<b>What it does:</b> Reads the raw text file from disk and loads it into a LangChain Document object ready for processing.
</p>

In [9]:
# Load the text file — replace 'sample_UK.txt' with your own file path
loader = TextLoader('sample_UK.txt')
documents = loader.load()

# Check how many documents were loaded (1 = the whole file as one document)
print(f" Documents loaded: {len(documents)}")

 Documents loaded: 1


<hr>
<h2 style="color:#28a745;"> Document Transformer — Text Splitter</h2>
<p style="color:#28a745; font-size:0.95rem;">
<b>What it does:</b> Splits the large document into smaller chunks so the LLM can process them within its token limit.
</p>
<ul style="color:#28a745;">
    <li><b>chunk_size=200</b> → each chunk is max 200 characters</li>
    <li><b>chunk_overlap=0</b> → no overlap between chunks</li>
</ul>

In [10]:
# Split the document into chunks of 200 characters
text_splitter = CharacterTextSplitter(
    chunk_size=200,     # Max characters per chunk
    chunk_overlap=0     # No overlap between consecutive chunks
)

texts = text_splitter.split_documents(documents)

# Check how many chunks were created
print(f" Total chunks created: {len(texts)}")

Created a chunk of size 411, which is longer than the specified 200
Created a chunk of size 355, which is longer than the specified 200
Created a chunk of size 361, which is longer than the specified 200
Created a chunk of size 436, which is longer than the specified 200
Created a chunk of size 401, which is longer than the specified 200
Created a chunk of size 361, which is longer than the specified 200
Created a chunk of size 336, which is longer than the specified 200
Created a chunk of size 254, which is longer than the specified 200
Created a chunk of size 363, which is longer than the specified 200
Created a chunk of size 360, which is longer than the specified 200
Created a chunk of size 279, which is longer than the specified 200
Created a chunk of size 370, which is longer than the specified 200
Created a chunk of size 277, which is longer than the specified 200
Created a chunk of size 349, which is longer than the specified 200
Created a chunk of size 320, which is longer tha

 Total chunks created: 18


In [11]:
# Preview all chunks
texts

[Document(metadata={'source': 'sample_UK.txt'}, page_content='The United Kingdom, officially known as the United Kingdom of Great Britain and Northern Ireland, is a sovereign country located off the northwestern coast of continental Europe. It is made up of four constituent countries: England, Scotland, Wales, and Northern Ireland. The UK is known for its rich history, cultural heritage, and significant global influence across politics, science, culture, and economics.'),
 Document(metadata={'source': 'sample_UK.txt'}, page_content='The United Kingdom covers an area of approximately 243,000 square kilometres and has a population of around 67 million people. London is the capital and largest city of the United Kingdom. It serves as the political, economic, and cultural centre of the country. Other major cities include Manchester, Birmingham, Edinburgh, Glasgow, Cardiff, and Belfast.'),
 Document(metadata={'source': 'sample_UK.txt'}, page_content='The United Kingdom is a constitutional m

<hr>
<h2 style="color:#e67e22;"> Text Embedding Model</h2>
<p style="color:#e67e22; font-size:0.95rem;">
<b>What it does:</b> Converts each text chunk into a vector — a list of 384 floating point numbers that represent the <b>meaning</b> of the text. Similar meanings produce similar vectors.
</p>
<p style="color:#e67e22;">
💡 We use <b>all-MiniLM-L6-v2</b> — a free, lightweight sentence transformer model that runs locally without needing an API key.
</p>

In [19]:
import os
from dotenv import load_dotenv
load_dotenv()

# Use OpenAI embeddings instead of SentenceTransformer
embeddings = OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))

print(" Embedding model loaded: OpenAI text-embedding-ada-002")

 Embedding model loaded: OpenAI text-embedding-ada-002


<hr>
<h2 style="color:#FA00EE;"> Vector Store — Chroma</h2>
<p style="color:#FA00EE; font-size:0.95rem;">
<b>What it does:</b> Takes all the chunks and their vectors and stores them in <b>Chroma</b> — an open-source, AI-native vector database. This makes them searchable by semantic similarity.
</p>
<p style="color:#FA00EE;">
 Chroma is free to use under Apache 2.0 license and runs entirely in memory — no setup required.
</p>

In [20]:
# Store all chunk embeddings in Chroma vector database
db = Chroma.from_documents(texts, embeddings)

print(" Vectors stored in Chroma database")

 Vectors stored in Chroma database


In [21]:
# Preview the raw numeric embeddings stored in Chroma
# Each chunk becomes a list of floating point numbers
db._collection.get(include=['embeddings'])

{'ids': ['e2734b14-eeea-4daf-af7c-bd5c4a541342',
  'd7059b20-f72e-4c41-abea-939f1db1f92c',
  '77f1c33a-a903-4fb7-b518-2099470c87d7',
  'b56ed83f-3a5a-4de9-8465-481b24179ad6',
  '2f421438-8aa0-42b2-834f-06e25688ce49',
  '819cd2a9-8efb-4063-918c-9987b7cc5d27',
  '50538128-0b95-42ad-a4df-46a1680bc9d6',
  '9cafd827-83b4-420d-8e24-c9dff127f9a8',
  '0858d155-b28f-4153-ad24-232968357d5e',
  '25698b1f-d146-4c22-a1ac-14198f3c26b7',
  '4c7f4f37-3d7d-4b2a-9434-5dedb5da3b26',
  '0e7f397f-3d3b-4ede-9f63-7e3682a771f4',
  'b37b3e1b-1626-4247-a6bc-f2924c0836dd',
  '630db3b2-0c57-441d-80e8-9fb7a9485640',
  '21785db2-0156-4866-b2b0-bd6dc66e193f',
  '5b52d988-5ef7-4c90-abf3-e2ae78b2f2fd',
  '8fc333a7-6bc8-428f-a19a-6f825aa95134',
  'f93328a8-b1b9-4499-93ee-da023f7040fa'],
 'embeddings': array([[ 0.00710655, -0.01737981, -0.00559432, ..., -0.00432331,
         -0.00604582, -0.00652516],
        [ 0.02386689,  0.00536025, -0.01242352, ...,  0.00851514,
         -0.02079164, -0.01391827],
        [ 0.005757

<hr>
<h2 style="color:#4BFA00;"> Retriever</h2>
<p style="color:#4BFA00; font-size:0.95rem;">
<b>What it does:</b> Wraps the vector store with a search interface. When given a question, it converts it to a vector and finds the <b>k most similar chunks</b> from the database.
</p>
<ul style="color:#4BFA00;">
    <li><b>k=2</b> → returns the top 2 most relevant chunks per query</li>
</ul>

In [22]:
# Create a retriever that returns the top 2 most relevant chunks
retriever = db.as_retriever(search_kwargs={"k": 2})

print(" Retriever ready — will return top 2 relevant chunks per query")

 Retriever ready — will return top 2 relevant chunks per query
